# UCTP CCB-TT for Thesis

## Thesis
`Creating no conflict and low fitness score timetable solution of UCTP Curriculum-based Timetable`

### Immport Python Library to use

In [ ]:
import pandas as pd
import numpy as np
import random
import datetime
from collections import defaultdict
import copy
import matplotlib.pyplot as plt
import mlflow
import os

print("Libraries imported successfully!")

### Dataset Course, Rooms, and Timeslots - Setup

In [ ]:
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta
import mlflow

def configure_courses(csv_path):
    df_courses = pd.read_csv(csv_path)
    df_courses['event_id'] = [f"EVT_{i:04d}" for i in range(len(df_courses))]
    df_courses['requires_practicum_room'] = df_courses['course_name'].str.contains('Praktikum', case=False, na=False)
    df_courses['slots_needed'] = df_courses['course_time_minute'] // 50
    
    cols_order = ['event_id', 'course_id', 'course_name', 'requires_practicum_room', 
                  'slots_needed', 'teacher_id', 'teacher_priority']
    available_cols = [c for c in cols_order if c in df_courses.columns]
    return df_courses[available_cols]

def configure_rooms_and_timeslots(rooms_csv_path):
    df_rooms = pd.read_csv(rooms_csv_path)
    
    theory_rooms = {'F2.1', 'F2.2', 'F2.4', 'F2.5', 'F2.6', 'F2.8', 'F2.9', 'F3.1', 'F3.10',
                    'F3.11', 'F3.12', 'F3.13', 'F3.14', 'F3.15', 'F3.16', 'F3.17', 'F3.18',
                    'F3.2', 'F3.3', 'F3.4', 'F3.5a', 'F3.5b', 'F3.6', 'F3.7a', 'F3.7b',
                    'F3.8', 'F3.9', 'F4.10', 'F4.11', 'F4.12', 'F4.13', 'F4.14', 'F4.2',
                    'F4.3', 'F4.4', 'F4.5', 'F4.6', 'F4.7', 'F4.9'}
    
    practicum_rooms = {'G1.2', 'G1.3', 'G1.4', 'G1.5', 'G1.6', 
                       'F2.1', 'F2.2', 'F2.4', 'F2.5', 'F2.6', 'F2.8', 'F2.9'}
    
    df_rooms['is_theory'] = df_rooms['room_id'].isin(theory_rooms)
    df_rooms['is_practicum'] = df_rooms['room_id'].isin(practicum_rooms)

    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    granularity_minutes = 50 
    all_timeslots = []
    slot_id_counter = 0
    
    for day in days:
        sessions = [
            {'type': 'Morning', 'start': "07:00", 'end': "12:00"},
            {'type': 'Afternoon', 'start': "12:30", 'end': "18:20"}
        ]
        for session in sessions:
            curr_time = datetime.strptime(session['start'], "%H:%M")
            end_time = datetime.strptime(session['end'], "%H:%M")
            while curr_time < end_time:
                all_timeslots.append({
                    'slot_id': f"T_{slot_id_counter:04d}",
                    'day': day,
                    'session_type': session['type'],
                    'time_start': curr_time.strftime("%H:%M"),
                    'time_end': (curr_time + timedelta(minutes=granularity_minutes)).strftime("%H:%M")
                })
                curr_time += timedelta(minutes=granularity_minutes)
                slot_id_counter += 1
                
    return df_rooms, pd.DataFrame(all_timeslots)

# Eksekusi Pembuatan Data
df_courses = configure_courses('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/mk_dsi_genap_praktikum.csv')
df_rooms, df_timeslots = configure_rooms_and_timeslots('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/rooms.csv')
print(f"Dataset Aktif -> Courses: {len(df_courses)} | Rooms: {len(df_rooms)} | Timeslots: {len(df_timeslots)}")

df_courses.to_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/mk_dsi_genap_praktikum_processed.csv', index=False)
df_rooms.to_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/rooms_processed.csv', index=False)
df_timeslots.to_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/timeslots_processed.csv', index=False)

### Fitness Function for Objective Calculation

In [ ]:
import numpy as np
import pandas as pd

def fitness_function(sol_x, mk_df, rooms_df, timeslots_df, valid_starts, rooms_prac, rooms_theo):
    penalty = 0
    assignments = []
    
    # State tracker untuk mencegah tumpang tindih secara absolut
    room_time_grid = {}
    teacher_time_grid = {}
    teacher_day_rooms = {}
    
    # Heuristic Decoder: Iterasi setiap mata kuliah
    for i in range(len(mk_df)):
        x = sol_x[i]
        req_prac = mk_df['requires_practicum_room'].iloc[i]
        teacher = mk_df['teacher_id'].iloc[i]
        priority = mk_df['teacher_priority'].iloc[i]
        
        valid_rooms = rooms_prac if req_prac else rooms_theo
        
        # 1. Himpun SEMUA pilihan (ruang, waktu) yang TIDAK BENTROK
        feasible_choices = []
        for r in valid_rooms:
            for start_idx in valid_starts:
                t1, t2 = start_idx, start_idx + 1
                
                # Cek ketersediaan ruang
                if (r, t1) in room_time_grid or (r, t2) in room_time_grid:
                    continue
                # Cek ketersediaan dosen
                if (teacher, t1) in teacher_time_grid or (teacher, t2) in teacher_time_grid:
                    continue
                
                feasible_choices.append((r, start_idx))
                
        # 2. Jika ada slot kosong, gunakan nilai x dari PO untuk memilih
        if len(feasible_choices) > 0:
            choice_idx = int(x * len(feasible_choices))
            if choice_idx >= len(feasible_choices): 
                choice_idx = len(feasible_choices) - 1
                
            selected_r, selected_start = feasible_choices[choice_idx]
            t1, t2 = selected_start, selected_start + 1
            day = timeslots_df['day'].iloc[selected_start]
            
            # Kunci slot agar tidak bisa dipakai course lain
            room_time_grid[(selected_r, t1)] = True
            room_time_grid[(selected_r, t2)] = True
            teacher_time_grid[(teacher, t1)] = True
            teacher_time_grid[(teacher, t2)] = True
            
            assignments.append({
                'course_i': i, 'room': selected_r, 
                'start_slot': selected_start, 'end_slot': t2, 
                'teacher': teacher, 'priority': priority, 'day': day
            })
            
            # --- Perhitungan Soft Constraint ---
            # A. Prioritas Sesi (Pagi/Siang)
            sess1 = timeslots_df['session_type'].iloc[t1]
            sess2 = timeslots_df['session_type'].iloc[t2]
            if priority == 1 and (sess1 != 'Morning' or sess2 != 'Morning'): 
                penalty += 10
            elif priority == 2 and (sess1 != 'Afternoon' or sess2 != 'Afternoon'): 
                penalty += 10
                
            # B. Perpindahan Ruang Dosen di Hari yang Sama
            if teacher not in teacher_day_rooms: 
                teacher_day_rooms[teacher] = {}
            if day not in teacher_day_rooms[teacher]: 
                teacher_day_rooms[teacher][day] = []
            teacher_day_rooms[teacher][day].append(selected_r)
            
        else:
            # Hard Constraint Violation sesungguhnya: Kelas gagal dijadwalkan karena grid penuh/buntu
            penalty += 100000  

    # Akumulasi Penalti Soft Constraint untuk perpindahan ruang
    for teacher, days in teacher_day_rooms.items():
        for day, rooms in days.items():
            unique_rooms = len(set(rooms))
            if unique_rooms > 1: 
                penalty += 50 * (unique_rooms - 1)
                
    return penalty, assignments

### Implement POA

In [ ]:
import numpy as np
import pandas as pd
import math

class Solution:
    def __init__(self, x, cost, assignments=None):
        self.X = x
        self.Cost = cost
        self.assignments = assignments

def exploration(Sol, lb, ub, dim, nSol, cost_func):
    Sol = sorted(Sol, key=lambda s: s.Cost)
    pCR = 0.20
    PCR = 1 - pCR
    p = PCR / nSol
    
    NewSol = [None] * nSol
    for i in range(nSol):
        x = Sol[i].X
        
        A = list(range(nSol))
        A.remove(i)
        np.random.shuffle(A)
        a, b, c, d, e, f = A[:6]
        
        G = 2 * np.random.rand() - 1
        
        if np.random.rand() < 0.5:
            y = np.random.rand(dim) * (ub - lb) + lb
        else:
            y = Sol[a].X + G * (Sol[a].X - Sol[b].X) + G * (((Sol[a].X - Sol[b].X) - (Sol[c].X - Sol[d].X)) + ((Sol[c].X - Sol[d].X) - (Sol[e].X - Sol[f].X)))
            
        y = np.clip(y, lb, ub)
        
        z = np.zeros(dim)
        j0 = np.random.randint(dim)
        for j in range(dim):
            if j == j0 or np.random.rand() <= pCR:
                z[j] = y[j]
            else:
                z[j] = x[j]
                
        cost, _ = cost_func(z)
        NewSol[i] = Solution(z, cost)
        
        if NewSol[i].Cost < Sol[i].Cost:
            Sol[i] = NewSol[i]
        else:
            pCR = pCR + p
            
    return Sol

def exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func):
    Q = 0.67
    Beta = 2
    
    NewSol = [None] * nSol
    for i in range(nSol):
        beta1 = 2 * np.random.rand()
        beta2 = np.random.randn(dim)
        w = np.random.randn(dim)
        v = np.random.randn(dim)
        
        F1 = np.random.randn(dim) * np.exp(2 - Iter * (2 / MaxIter))
        F2 = w * (v**2) * np.cos((2 * np.random.rand()) * w)
        
        mbest = np.mean([s.X for s in Sol], axis=0)
        R_1 = 2 * np.random.rand() - 1
        S1 = (2 * np.random.rand() - 1 + np.random.randn(dim))
        S2 = (F1 * R_1 * Sol[i].X + F2 * (1 - R_1) * Best.X)
        VEC = S2 / (S1 + 1e-10) # Menghindari divisi oleh nol
        
        if np.random.rand() <= 0.5:
            Xatack = VEC
            if np.random.rand() > Q:
                r_idx = np.random.randint(nSol)
                x_new = Best.X + beta1 * (np.exp(beta2)) * (Sol[r_idx].X - Sol[i].X)
            else:
                x_new = beta1 * Xatack - Best.X
        else:
            r1 = np.random.randint(nSol)
            sign_part = (-1) ** np.random.randint(0, 2)
            x_new = (mbest * Sol[r1].X - sign_part * Sol[i].X) / (1 + (Beta * np.random.rand()))
            
        x_new = np.clip(x_new, lb, ub)
        cost, _ = cost_func(x_new)
        NewSol[i] = Solution(x_new, cost)
        
        if NewSol[i].Cost < Sol[i].Cost:
            Sol[i] = NewSol[i]
            
    return Sol

def puma_optimizer(nSol, MaxIter, lb, ub, dim, cost_func):
    Sol = []
    for i in range(nSol):
        x = lb + (ub - lb) * np.random.rand(dim)
        cost, _ = cost_func(x)
        Sol.append(Solution(x, cost))
        
    Best = min(Sol, key=lambda s: s.Cost)
    Initial_Best = Solution(Best.X.copy(), Best.Cost)
    
    UnSelected = [1, 1]
    F3_Explore, F3_Exploit = 0, 0
    Seq_Time_Explore = [1, 1, 1]
    Seq_Time_Exploit = [1, 1, 1]
    Seq_Cost_Explore = [1, 1, 1]
    Seq_Cost_Exploit = [1, 1, 1]
    PF = [0.5, 0.5, 0.3]
    PF_F3 = []
    Mega_Explor, Mega_Exploit = 0.99, 0.99
    Costs_Explor, Costs_Exploit = np.zeros(3), np.zeros(3)
    Flag_Change = 1
    
    # Fase Unexperienced
    for Iter in range(1, 4):
        Sol_Explor = exploration([Solution(s.X.copy(), s.Cost) for s in Sol], lb, ub, dim, nSol, cost_func)
        Costs_Explor[Iter-1] = min([s.Cost for s in Sol_Explor])
        
        Sol_Exploit = exploitation([Solution(s.X.copy(), s.Cost) for s in Sol], lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func)
        Costs_Exploit[Iter-1] = min([s.Cost for s in Sol_Exploit])
        
        Sol = sorted(Sol + Sol_Explor + Sol_Exploit, key=lambda s: s.Cost)[:nSol]
        Best = Sol[0]
        
    Seq_Cost_Explore[0] = abs(Initial_Best.Cost - Costs_Explor[0])
    Seq_Cost_Exploit[0] = abs(Initial_Best.Cost - Costs_Exploit[0])
    Seq_Cost_Explore[1] = abs(Costs_Explor[1] - Costs_Explor[0])
    Seq_Cost_Exploit[1] = abs(Costs_Exploit[1] - Costs_Exploit[0])
    Seq_Cost_Explore[2] = abs(Costs_Explor[2] - Costs_Explor[1])
    Seq_Cost_Exploit[2] = abs(Costs_Exploit[2] - Costs_Exploit[1])
    
    for i in range(3):
        if Seq_Cost_Explore[i] != 0: PF_F3.append(Seq_Cost_Explore[i])
        if Seq_Cost_Exploit[i] != 0: PF_F3.append(Seq_Cost_Exploit[i])
        
    F1_Explor = PF[0] * (Seq_Cost_Explore[0] / Seq_Time_Explore[0])
    F1_Exploit = PF[0] * (Seq_Cost_Exploit[0] / Seq_Time_Exploit[0])
    F2_Explor = PF[1] * (sum(Seq_Cost_Explore) / sum(Seq_Time_Explore))
    F2_Exploit = PF[1] * (sum(Seq_Cost_Exploit) / sum(Seq_Time_Exploit))
    
    Score_Explore = (PF[0] * F1_Explor) + (PF[1] * F2_Explor)
    Score_Exploit = (PF[0] * F1_Exploit) + (PF[1] * F2_Exploit)
    
    # Fase Experienced
    for Iter in range(4, MaxIter + 1):
        if Score_Explore > Score_Exploit:
            SelectFlag = 1
            Sol = exploration(Sol, lb, ub, dim, nSol, cost_func)
            Count_select = UnSelected.copy()
            UnSelected[1] += 1
            UnSelected[0] = 1
            F3_Explore = PF[2]
            F3_Exploit += PF[2]
            
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Explore[2], Seq_Cost_Explore[1] = Seq_Cost_Explore[1], Seq_Cost_Explore[0]
            Seq_Cost_Explore[0] = abs(Best.Cost - TBest.Cost)
            
            if Seq_Cost_Explore[0] != 0: PF_F3.append(Seq_Cost_Explore[0])
            if TBest.Cost < Best.Cost: Best = TBest
            
        else:
            SelectFlag = 2
            Sol = exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func)
            Count_select = UnSelected.copy()
            UnSelected[0] += 1
            UnSelected[1] = 1
            F3_Explore += PF[2]
            F3_Exploit = PF[2]
            
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Exploit[2], Seq_Cost_Exploit[1] = Seq_Cost_Exploit[1], Seq_Cost_Exploit[0]
            Seq_Cost_Exploit[0] = abs(Best.Cost - TBest.Cost)
            
            if Seq_Cost_Exploit[0] != 0: PF_F3.append(Seq_Cost_Exploit[0])
            if TBest.Cost < Best.Cost: Best = TBest
            
        if Flag_Change != SelectFlag:
            Flag_Change = SelectFlag
            Seq_Time_Explore[2], Seq_Time_Explore[1] = Seq_Time_Explore[1], Seq_Time_Explore[0]
            Seq_Time_Explore[0] = Count_select[0]
            Seq_Time_Exploit[2], Seq_Time_Exploit[1] = Seq_Time_Exploit[1], Seq_Time_Exploit[0]
            Seq_Time_Exploit[0] = Count_select[1]
            
        F1_Explor = PF[0] * (Seq_Cost_Explore[0] / Seq_Time_Explore[0])
        F1_Exploit = PF[0] * (Seq_Cost_Exploit[0] / Seq_Time_Exploit[0])
        F2_Explor = PF[1] * (sum(Seq_Cost_Explore) / sum(Seq_Time_Explore))
        F2_Exploit = PF[1] * (sum(Seq_Cost_Exploit) / sum(Seq_Time_Exploit))
        
        if Score_Explore < Score_Exploit:
            Mega_Explor = max((Mega_Explor - 0.01), 0.01)
            Mega_Exploit = 0.99
        elif Score_Explore > Score_Exploit:
            Mega_Explor = 0.99
            Mega_Exploit = max((Mega_Exploit - 0.01), 0.01)
            
        lmn_Explore, lmn_Exploit = 1 - Mega_Explor, 1 - Mega_Exploit
        
        min_PF_F3 = min(PF_F3) if PF_F3 else 0
        Score_Explore = (Mega_Explor * F1_Explor) + (Mega_Explor * F2_Explor) + (lmn_Explore * (min_PF_F3 * F3_Explore))
        Score_Exploit = (Mega_Exploit * F1_Exploit) + (Mega_Exploit * F2_Exploit) + (lmn_Exploit * (min_PF_F3 * F3_Exploit))
        
    return Best

### Run & Save POA - Logs and Timetable

In [ ]:
import numpy as np
import pandas as pd
from puma_optimizer import puma_optimizer # Pastikan file puma_optimizer.py berada di direktori yang sama

# Muat data
mk_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/mk_dsi_genap_praktikum_processed.csv')
rooms_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/rooms_processed.csv')
timeslots_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/timeslots_processed.csv')

# Identifikasi blok waktu (2 SKS berturut-turut tanpa melanggar transisi hari/sesi)
valid_starts = []
for i in range(len(timeslots_df) - 1):
    t1 = timeslots_df.iloc[i]
    t2 = timeslots_df.iloc[i+1]
    if t1['day'] == t2['day'] and t1['time_end'] == t2['time_start']:
        valid_starts.append(i)

rooms_prac = rooms_df[rooms_df['is_practicum'] == True].index.tolist()
rooms_theo = rooms_df[rooms_df['is_theory'] == True].index.tolist()

def fitness_function(sol_x):
    penalty = 0
    assignments = []
    
    # State tracker diinisialisasi ulang untuk setiap kandidat solusi (sol_x)
    room_time_grid = {}
    teacher_time_grid = {}
    teacher_day_rooms = {}
    
    # Heuristic Decoder: Iterasi komputasi secara skuensial berdasarkan indeks course
    for i in range(len(mk_df)):
        x = sol_x[i]
        req_prac = mk_df['requires_practicum_room'].iloc[i]
        teacher = mk_df['teacher_id'].iloc[i]
        priority = mk_df['teacher_priority'].iloc[i]
        
        valid_rooms = rooms_prac if req_prac else rooms_theo
        
        # 1. Himpun SEMUA pilihan (ruang, waktu) yang 100% TIDAK BENTROK
        feasible_choices = []
        for r in valid_rooms:
            for start_idx in valid_starts:
                t1, t2 = start_idx, start_idx + 1
                
                # Pruning ruang: Cek ketersediaan ruang pada T1 dan T2
                if (r, t1) in room_time_grid or (r, t2) in room_time_grid:
                    continue
                # Pruning dosen: Cek ketersediaan dosen pada T1 dan T2
                if (teacher, t1) in teacher_time_grid or (teacher, t2) in teacher_time_grid:
                    continue
                
                feasible_choices.append((r, start_idx))
                
        # 2. Transformasi probabilitas: Jika ada slot kosong, gunakan nilai x dari PO untuk mapping
        if len(feasible_choices) > 0:
            choice_idx = int(x * len(feasible_choices))
            if choice_idx >= len(feasible_choices): 
                choice_idx = len(feasible_choices) - 1
                
            selected_r, selected_start = feasible_choices[choice_idx]
            t1, t2 = selected_start, selected_start + 1
            day = timeslots_df['day'].iloc[selected_start]
            
            # Kunci slot (Update State) agar tidak memicu over-booking pada course berikutnya
            room_time_grid[(selected_r, t1)] = True
            room_time_grid[(selected_r, t2)] = True
            teacher_time_grid[(teacher, t1)] = True
            teacher_time_grid[(teacher, t2)] = True
            
            assignments.append({
                'course_i': i, 'room': selected_r, 
                'start_slot': selected_start, 'end_slot': t2, 
                'teacher': teacher, 'priority': priority, 'day': day
            })
            
            # --- Evaluasi Soft Constraint Parsial ---
            # A. Evaluasi Prioritas Sesi Mengajar
            sess1 = timeslots_df['session_type'].iloc[t1]
            sess2 = timeslots_df['session_type'].iloc[t2]
            if priority == 1 and (sess1 != 'Morning' or sess2 != 'Morning'): 
                penalty += 10
            elif priority == 2 and (sess1 != 'Afternoon' or sess2 != 'Afternoon'): 
                penalty += 10
                
            # B. Perekaman data untuk evaluasi perpindahan ruang dosen
            if teacher not in teacher_day_rooms: 
                teacher_day_rooms[teacher] = {}
            if day not in teacher_day_rooms[teacher]: 
                teacher_day_rooms[teacher][day] = []
            teacher_day_rooms[teacher][day].append(selected_r)
            
        else:
            # Fatal Hard Constraint Violation: Terjadi jika kapasitas semesta penjadwalan tidak mencukupi
            penalty += 100000  

    # --- Evaluasi Soft Constraint Akumulatif ---
    # Kalkulasi penalti untuk perpindahan ruang dosen pada hari yang sama
    for teacher, days in teacher_day_rooms.items():
        for day, rooms in days.items():
            unique_rooms = len(set(rooms))
            if unique_rooms > 1: 
                penalty += 50 * (unique_rooms - 1)
                
    return penalty, assignments


# ==========================================
# Parameter Komputasi dan Eksekusi Algoritma
# ==========================================
nSol = 30     # Ukuran populasi dinaikkan untuk meningkatkan derajat eksplorasi
MaxIter = 50  # Batas iterasi harus lebih moderat (Min. 50 untuk konvergensi soft-constraint)
dim = len(mk_df)
lb, ub = 0.0, 1.0

# Asumsikan Anda memanggil fungsi Puma Optimizer yang sudah didefinisikan secara modular
# Pastikan puma_optimizer mengembalikan objek kandidat terbaik
best_sol, _ = puma_optimizer(nSol, MaxIter, lb, ub, dim, fitness_function)

# Ekstraksi matriks assignments dari solusi terbaik
_, final_assignments = fitness_function(best_sol.X)

# Transformasi output ke dalam data tabular
res_list = []
for a in final_assignments:
    course = mk_df.iloc[a['course_i']]
    room = rooms_df.iloc[a['room']]
    t1 = timeslots_df.iloc[a['start_slot']]
    t2 = timeslots_df.iloc[a['end_slot']]
    res_list.append({
        'event_id': course['event_id'], 
        'course_name': course['course_name'],
        'teacher_id': course['teacher_id'], 
        'room_id': room['room_id'],
        'day': t1['day'], 
        'time_start': t1['time_start'], 
        'time_end': t2['time_end']
    })

res_df = pd.DataFrame(res_list)
res_df.to_csv('scheduled_courses.csv', index=False)

### Evaluation - Convergence & Stats

In [ ]:
print("Hello, World!")

### All in one

In [ ]:
import numpy as np
import pandas as pd
import time

# ==========================================
# 1. Deklarasi Struktur Data & Load Data
# ==========================================
mk_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/mk_dsi_genap_praktikum_processed.csv')
rooms_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/rooms_processed.csv')
timeslots_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/timeslots_processed.csv')

valid_starts = []
for i in range(len(timeslots_df) - 1):
    t1 = timeslots_df.iloc[i]
    t2 = timeslots_df.iloc[i+1]
    if t1['day'] == t2['day'] and t1['time_end'] == t2['time_start']:
        valid_starts.append(i)

rooms_prac = rooms_df[rooms_df['is_practicum'] == True].index.tolist()
rooms_theo = rooms_df[rooms_df['is_theory'] == True].index.tolist()

# ==========================================
# 2. Heuristic Decoder & Objective Function
# ==========================================
def fitness_function(sol_x):
    assignments = []
    
    # Metrik Terdekomposisi
    hc_count, hc_value = 0, 0
    sc_count, sc_value = 0, 0
    
    room_time_grid = {}
    teacher_time_grid = {}
    teacher_day_rooms = {}
    
    for i in range(len(mk_df)):
        x = sol_x[i]
        req_prac = mk_df['requires_practicum_room'].iloc[i]
        teacher = mk_df['teacher_id'].iloc[i]
        priority = mk_df['teacher_priority'].iloc[i]
        
        valid_rooms = rooms_prac if req_prac else rooms_theo
        
        feasible_choices = []
        for r in valid_rooms:
            for start_idx in valid_starts:
                t1, t2 = start_idx, start_idx + 1
                if (r, t1) in room_time_grid or (r, t2) in room_time_grid: continue
                if (teacher, t1) in teacher_time_grid or (teacher, t2) in teacher_time_grid: continue
                feasible_choices.append((r, start_idx))
                
        if len(feasible_choices) > 0:
            choice_idx = int(x * len(feasible_choices))
            if choice_idx >= len(feasible_choices): choice_idx = len(feasible_choices) - 1
                
            selected_r, selected_start = feasible_choices[choice_idx]
            t1, t2 = selected_start, selected_start + 1
            day = timeslots_df['day'].iloc[selected_start]
            
            room_time_grid[(selected_r, t1)] = True
            room_time_grid[(selected_r, t2)] = True
            teacher_time_grid[(teacher, t1)] = True
            teacher_time_grid[(teacher, t2)] = True
            
            assignments.append({
                'course_i': i, 'room': selected_r, 
                'start_slot': selected_start, 'end_slot': t2, 
                'teacher': teacher, 'priority': priority, 'day': day
            })
            
            # Soft Constraint: Prioritas Mengajar
            sess1, sess2 = timeslots_df['session_type'].iloc[t1], timeslots_df['session_type'].iloc[t2]
            if priority == 1 and (sess1 != 'Morning' or sess2 != 'Morning'): 
                sc_count += 1; sc_value += 10
            elif priority == 2 and (sess1 != 'Afternoon' or sess2 != 'Afternoon'): 
                sc_count += 1; sc_value += 10
                
            # Pelacakan Perpindahan Ruang
            if teacher not in teacher_day_rooms: teacher_day_rooms[teacher] = {}
            if day not in teacher_day_rooms[teacher]: teacher_day_rooms[teacher][day] = []
            teacher_day_rooms[teacher][day].append(selected_r)
            
        else:
            # Fatal Hard Constraint Violation
            hc_count += 1; hc_value += 100000  

    # Soft Constraint: Kalkulasi Penalti Perpindahan Ruang
    for teacher, days in teacher_day_rooms.items():
        for day, rooms in days.items():
            unique_rooms = len(set(rooms))
            if unique_rooms > 1: 
                violations = unique_rooms - 1
                sc_count += violations
                sc_value += 50 * violations
                
    total_penalty = hc_value + sc_value
    metrics = {
        'hc_count': hc_count, 'hc_value': hc_value,
        'sc_count': sc_count, 'sc_value': sc_value
    }
                
    return total_penalty, assignments, metrics

# ==========================================
# 3. Komponen Puma Optimizer
# ==========================================
class Solution:
    def __init__(self, x, cost, assignments=None, metrics=None):
        self.X = x
        self.Cost = cost
        self.assignments = assignments
        self.metrics = metrics

def print_iteration_log(Iter, Sol, Best):
    worst_cost = max([s.Cost for s in Sol])
    metrics = Best.metrics
    
    # Format string output yang presisi dan rata untuk keperluan analisis log console
    print(f"Iter: {Iter:03d} | "
          f"Best Fitness: {Best.Cost:8.0f} | "
          f"Worst Fitness: {worst_cost:8.0f} || "
          f"HC Violations: {metrics['hc_count']:3d} (Val: {metrics['hc_value']:7.0f}) | "
          f"SC Violations: {metrics['sc_count']:3d} (Val: {metrics['sc_value']:5.0f})")

def exploration(Sol, lb, ub, dim, nSol, cost_func):
    Sol = sorted(Sol, key=lambda s: s.Cost)
    pCR = 0.20
    PCR = 1 - pCR
    p = PCR / nSol
    
    NewSol = [None] * nSol
    for i in range(nSol):
        x = Sol[i].X
        A = list(range(nSol)); A.remove(i); np.random.shuffle(A)
        a, b, c, d, e, f = A[:6]
        
        G = 2 * np.random.rand() - 1
        if np.random.rand() < 0.5:
            y = np.random.rand(dim) * (ub - lb) + lb
        else:
            y = Sol[a].X + G * (Sol[a].X - Sol[b].X) + G * (((Sol[a].X - Sol[b].X) - (Sol[c].X - Sol[d].X)) + ((Sol[c].X - Sol[d].X) - (Sol[e].X - Sol[f].X)))
            
        y = np.clip(y, lb, ub)
        z = np.zeros(dim)
        j0 = np.random.randint(dim)
        for j in range(dim):
            z[j] = y[j] if (j == j0 or np.random.rand() <= pCR) else x[j]
                
        cost, asg, met = cost_func(z)
        NewSol[i] = Solution(z, cost, asg, met)
        
        if NewSol[i].Cost < Sol[i].Cost:
            Sol[i] = NewSol[i]
        else:
            pCR = pCR + p
    return Sol

def exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func):
    Q, Beta = 0.67, 2
    NewSol = [None] * nSol
    for i in range(nSol):
        beta1, beta2 = 2 * np.random.rand(), np.random.randn(dim)
        w, v = np.random.randn(dim), np.random.randn(dim)
        F1 = np.random.randn(dim) * np.exp(2 - Iter * (2 / MaxIter))
        F2 = w * (v**2) * np.cos((2 * np.random.rand()) * w)
        
        mbest = np.mean([s.X for s in Sol], axis=0)
        R_1 = 2 * np.random.rand() - 1
        S1 = (2 * np.random.rand() - 1 + np.random.randn(dim))
        S2 = (F1 * R_1 * Sol[i].X + F2 * (1 - R_1) * Best.X)
        VEC = S2 / (S1 + 1e-10) 
        
        if np.random.rand() <= 0.5:
            if np.random.rand() > Q:
                r_idx = np.random.randint(nSol)
                x_new = Best.X + beta1 * (np.exp(beta2)) * (Sol[r_idx].X - Sol[i].X)
            else:
                x_new = beta1 * VEC - Best.X
        else:
            r1 = np.random.randint(nSol)
            sign_part = (-1) ** np.random.randint(0, 2)
            x_new = (mbest * Sol[r1].X - sign_part * Sol[i].X) / (1 + (Beta * np.random.rand()))
            
        x_new = np.clip(x_new, lb, ub)
        cost, asg, met = cost_func(x_new)
        NewSol[i] = Solution(x_new, cost, asg, met)
        
        if NewSol[i].Cost < Sol[i].Cost:
            Sol[i] = NewSol[i]
    return Sol

def run_puma_optimizer(nSol, MaxIter, lb, ub, dim, cost_func):
    Sol = []
    print("Mengevaluasi populasi inisial...")
    for i in range(nSol):
        x = lb + (ub - lb) * np.random.rand(dim)
        cost, asg, met = cost_func(x)
        Sol.append(Solution(x, cost, asg, met))
        
    Best = min(Sol, key=lambda s: s.Cost)
    Initial_Best = Solution(Best.X.copy(), Best.Cost, Best.assignments, Best.metrics)
    
    # Inisialisasi hyper-parameter adaptif PO
    UnSelected = [1, 1]
    F3_Explore, F3_Exploit = 0, 0
    Seq_Time_Explore, Seq_Time_Exploit = [1, 1, 1], [1, 1, 1]
    Seq_Cost_Explore, Seq_Cost_Exploit = [1, 1, 1], [1, 1, 1]
    PF = [0.5, 0.5, 0.3]
    PF_F3 = []
    Mega_Explor, Mega_Exploit = 0.99, 0.99
    Costs_Explor, Costs_Exploit = np.zeros(3), np.zeros(3)
    Flag_Change = 1
    
    print("\n" + "="*90)
    print("MEMULAI PROSES OPTIMASI PUMA".center(90))
    print("="*90)

    # ---------------- Unexperienced Phase ----------------
    for Iter in range(1, 4):
        Sol_Explor = exploration([Solution(s.X.copy(), s.Cost, s.assignments, s.metrics) for s in Sol], lb, ub, dim, nSol, cost_func)
        Costs_Explor[Iter-1] = min([s.Cost for s in Sol_Explor])
        
        Sol_Exploit = exploitation([Solution(s.X.copy(), s.Cost, s.assignments, s.metrics) for s in Sol], lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func)
        Costs_Exploit[Iter-1] = min([s.Cost for s in Sol_Exploit])
        
        Sol = sorted(Sol + Sol_Explor + Sol_Exploit, key=lambda s: s.Cost)[:nSol]
        Best = Sol[0]
        
        print_iteration_log(Iter, Sol, Best)
        
    Seq_Cost_Explore[0] = abs(Initial_Best.Cost - Costs_Explor[0])
    Seq_Cost_Exploit[0] = abs(Initial_Best.Cost - Costs_Exploit[0])
    Seq_Cost_Explore[1] = abs(Costs_Explor[1] - Costs_Explor[0])
    Seq_Cost_Exploit[1] = abs(Costs_Exploit[1] - Costs_Exploit[0])
    Seq_Cost_Explore[2] = abs(Costs_Explor[2] - Costs_Explor[1])
    Seq_Cost_Exploit[2] = abs(Costs_Exploit[2] - Costs_Exploit[1])
    
    for i in range(3):
        if Seq_Cost_Explore[i] != 0: PF_F3.append(Seq_Cost_Explore[i])
        if Seq_Cost_Exploit[i] != 0: PF_F3.append(Seq_Cost_Exploit[i])
        
    # ---------------- Experienced Phase ----------------
    for Iter in range(4, MaxIter + 1):
        # Perhitungan dinamis mekanisme switch (Exploration vs Exploitation)
        F1_Explor = PF[0] * (Seq_Cost_Explore[0] / Seq_Time_Explore[0])
        F1_Exploit = PF[0] * (Seq_Cost_Exploit[0] / Seq_Time_Exploit[0])
        F2_Explor = PF[1] * (sum(Seq_Cost_Explore) / sum(Seq_Time_Explore))
        F2_Exploit = PF[1] * (sum(Seq_Cost_Exploit) / sum(Seq_Time_Exploit))
        Score_Explore = (Mega_Explor * F1_Explor) + (Mega_Explor * F2_Explor)
        Score_Exploit = (Mega_Exploit * F1_Exploit) + (Mega_Exploit * F2_Exploit)

        if Score_Explore > Score_Exploit:
            SelectFlag = 1
            Sol = exploration(Sol, lb, ub, dim, nSol, cost_func)
            Count_select = UnSelected.copy()
            UnSelected[1] += 1; UnSelected[0] = 1
            F3_Explore = PF[2]; F3_Exploit += PF[2]
            
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Explore[2], Seq_Cost_Explore[1] = Seq_Cost_Explore[1], Seq_Cost_Explore[0]
            Seq_Cost_Explore[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Explore[0] != 0: PF_F3.append(Seq_Cost_Explore[0])
            if TBest.Cost < Best.Cost: Best = TBest
        else:
            SelectFlag = 2
            Sol = exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func)
            Count_select = UnSelected.copy()
            UnSelected[0] += 1; UnSelected[1] = 1
            F3_Explore += PF[2]; F3_Exploit = PF[2]
            
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Exploit[2], Seq_Cost_Exploit[1] = Seq_Cost_Exploit[1], Seq_Cost_Exploit[0]
            Seq_Cost_Exploit[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Exploit[0] != 0: PF_F3.append(Seq_Cost_Exploit[0])
            if TBest.Cost < Best.Cost: Best = TBest
            
        if Flag_Change != SelectFlag:
            Flag_Change = SelectFlag
            Seq_Time_Explore[2], Seq_Time_Explore[1], Seq_Time_Explore[0] = Seq_Time_Explore[1], Seq_Time_Explore[0], Count_select[0]
            Seq_Time_Exploit[2], Seq_Time_Exploit[1], Seq_Time_Exploit[0] = Seq_Time_Exploit[1], Seq_Time_Exploit[0], Count_select[1]
            
        if Score_Explore < Score_Exploit:
            Mega_Explor = max((Mega_Explor - 0.01), 0.01); Mega_Exploit = 0.99
        elif Score_Explore > Score_Exploit:
            Mega_Explor = 0.99; Mega_Exploit = max((Mega_Exploit - 0.01), 0.01)
            
        print_iteration_log(Iter, Sol, Best)
        
    return Best

# ==========================================
# 4. Eksekusi Program
# ==========================================
nSol = 30     
MaxIter = 50  
dim = len(mk_df)
lb, ub = 0.0, 1.0

best_solution = run_puma_optimizer(nSol, MaxIter, lb, ub, dim, fitness_function)

# Ekspor hasil solusi akhir ke format CSV
res_list = []
for a in best_solution.assignments:
    course = mk_df.iloc[a['course_i']]
    room = rooms_df.iloc[a['room']]
    t1 = timeslots_df.iloc[a['start_slot']]
    t2 = timeslots_df.iloc[a['end_slot']]
    res_list.append({
        'event_id': course['event_id'], 'course_name': course['course_name'],
        'teacher_id': course['teacher_id'], 'room_id': room['room_id'],
        'day': t1['day'], 'time_start': t1['time_start'], 'time_end': t2['time_end']
    })

res_df = pd.DataFrame(res_list)
res_df.to_csv('scheduled_courses.csv', index=False)
print("\n[INFO] Solusi terbaik berhasil disimpan di 'scheduled_courses.csv'")

### Heuristic Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import time

# ==========================================
# 1. Deklarasi & Heuristic Preprocessing
# ==========================================
print("[INFO] Memuat dan memproses dataset...")
mk_df_raw = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/mk_dsi_genap_praktikum_processed.csv')
rooms_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/rooms_processed.csv')
timeslots_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/timeslots_processed.csv')

valid_starts = []
for i in range(len(timeslots_df) - 1):
    t1 = timeslots_df.iloc[i]
    t2 = timeslots_df.iloc[i+1]
    if t1['day'] == t2['day'] and t1['time_end'] == t2['time_start']:
        valid_starts.append(i)

rooms_prac = rooms_df[rooms_df['is_practicum'] == True].index.tolist()
rooms_theo = rooms_df[rooms_df['is_theory'] == True].index.tolist()

# --- IMPLEMENTASI GRAPH COLORING HEURISTIC (Largest Degree First) ---
# Menghitung beban mengajar dosen (semakin tinggi, semakin sulit dijadwalkan)
teacher_workload = mk_df_raw['teacher_id'].value_counts().to_dict()
mk_df_raw['teacher_degree'] = mk_df_raw['teacher_id'].map(teacher_workload)

# Mengurutkan dataset: 
# 1. Praktikum (True didahulukan karena ruang terbatas)
# 2. Beban Dosen (Tertinggi didahulukan)
mk_df_sorted = mk_df_raw.sort_values(
    by=['requires_practicum_room', 'teacher_degree'], 
    ascending=[False, False]
).reset_index() # reset_index menyimpan index asli pada kolom 'index'

# ==========================================
# 2. Heuristic Decoder & Objective Function
# ==========================================
def fitness_function(sol_x):
    assignments = []
    hc_count, hc_value = 0, 0
    sc_count, sc_value = 0, 0
    
    room_time_grid = {}
    teacher_time_grid = {}
    teacher_day_rooms = {}
    
    # Evaluasi dilakukan pada DataFrame yang sudah diurutkan secara Heuristik
    for idx in range(len(mk_df_sorted)):
        x = sol_x[idx] # Dimensi PO kini berkorespondensi dengan urutan terurut
        
        # Ambil data dari dataset terurut
        original_course_idx = mk_df_sorted['index'].iloc[idx]
        req_prac = mk_df_sorted['requires_practicum_room'].iloc[idx]
        teacher = mk_df_sorted['teacher_id'].iloc[idx]
        priority = mk_df_sorted['teacher_priority'].iloc[idx]
        
        valid_rooms = rooms_prac if req_prac else rooms_theo
        
        feasible_choices = []
        for r in valid_rooms:
            for start_idx in valid_starts:
                t1, t2 = start_idx, start_idx + 1
                if (r, t1) in room_time_grid or (r, t2) in room_time_grid: continue
                if (teacher, t1) in teacher_time_grid or (teacher, t2) in teacher_time_grid: continue
                feasible_choices.append((r, start_idx))
                
        if len(feasible_choices) > 0:
            # Seleksi menggunakan nilai probabilitas dari Puma Optimizer
            choice_idx = int(x * len(feasible_choices))
            if choice_idx >= len(feasible_choices): choice_idx = len(feasible_choices) - 1
                
            selected_r, selected_start = feasible_choices[choice_idx]
            t1, t2 = selected_start, selected_start + 1
            day = timeslots_df['day'].iloc[selected_start]
            
            # Kunci status State Tracker
            room_time_grid[(selected_r, t1)] = True
            room_time_grid[(selected_r, t2)] = True
            teacher_time_grid[(teacher, t1)] = True
            teacher_time_grid[(teacher, t2)] = True
            
            # Simpan assignment dengan referensi original_course_idx
            assignments.append({
                'course_i': original_course_idx, 'room': selected_r, 
                'start_slot': selected_start, 'end_slot': t2, 
                'teacher': teacher, 'priority': priority, 'day': day
            })
            
            # Penalti Soft Constraint: Prioritas Mengajar
            sess1, sess2 = timeslots_df['session_type'].iloc[t1], timeslots_df['session_type'].iloc[t2]
            if priority == 1 and (sess1 != 'Morning' or sess2 != 'Morning'): 
                sc_count += 1; sc_value += 10
            elif priority == 2 and (sess1 != 'Afternoon' or sess2 != 'Afternoon'): 
                sc_count += 1; sc_value += 10
                
            # Pelacakan Soft Constraint: Perpindahan Ruang
            if teacher not in teacher_day_rooms: teacher_day_rooms[teacher] = {}
            if day not in teacher_day_rooms[teacher]: teacher_day_rooms[teacher][day] = []
            teacher_day_rooms[teacher][day].append(selected_r)
            
        else:
            # Fatal Hard Constraint Violation
            hc_count += 1; hc_value += 100000  

    # Penalti Soft Constraint: Kalkulasi Penalti Perpindahan Ruang
    for teacher, days in teacher_day_rooms.items():
        for day, rooms in days.items():
            unique_rooms = len(set(rooms))
            if unique_rooms > 1: 
                violations = unique_rooms - 1
                sc_count += violations
                sc_value += 50 * violations
                
    total_penalty = hc_value + sc_value
    metrics = {
        'hc_count': hc_count, 'hc_value': hc_value,
        'sc_count': sc_count, 'sc_value': sc_value
    }
                
    return total_penalty, assignments, metrics

# ==========================================
# 3. Komponen Puma Optimizer
# ==========================================
class Solution:
    def __init__(self, x, cost, assignments=None, metrics=None):
        self.X = x
        self.Cost = cost
        self.assignments = assignments
        self.metrics = metrics

def print_iteration_log(Iter, Sol, Best):
    worst_cost = max([s.Cost for s in Sol])
    metrics = Best.metrics
    
    # Format string output yang presisi dan rata untuk keperluan analisis log console
    print(f"Iter: {Iter:03d} | "
          f"Best Fitness: {Best.Cost:8.0f} | "
          f"Worst Fitness: {worst_cost:8.0f} || "
          f"HC Violations: {metrics['hc_count']:3d} (Val: {metrics['hc_value']:7.0f}) | "
          f"SC Violations: {metrics['sc_count']:3d} (Val: {metrics['sc_value']:5.0f})")

def exploration(Sol, lb, ub, dim, nSol, cost_func):
    Sol = sorted(Sol, key=lambda s: s.Cost)
    pCR = 0.20
    PCR = 1 - pCR
    p = PCR / nSol
    
    NewSol = [None] * nSol
    for i in range(nSol):
        x = Sol[i].X
        A = list(range(nSol)); A.remove(i); np.random.shuffle(A)
        a, b, c, d, e, f = A[:6]
        
        G = 2 * np.random.rand() - 1
        if np.random.rand() < 0.5:
            y = np.random.rand(dim) * (ub - lb) + lb
        else:
            y = Sol[a].X + G * (Sol[a].X - Sol[b].X) + G * (((Sol[a].X - Sol[b].X) - (Sol[c].X - Sol[d].X)) + ((Sol[c].X - Sol[d].X) - (Sol[e].X - Sol[f].X)))
            
        y = np.clip(y, lb, ub)
        z = np.zeros(dim)
        j0 = np.random.randint(dim)
        for j in range(dim):
            z[j] = y[j] if (j == j0 or np.random.rand() <= pCR) else x[j]
                
        cost, asg, met = cost_func(z)
        NewSol[i] = Solution(z, cost, asg, met)
        
        if NewSol[i].Cost < Sol[i].Cost:
            Sol[i] = NewSol[i]
        else:
            pCR = pCR + p
    return Sol

def exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func):
    Q, Beta = 0.67, 2
    NewSol = [None] * nSol
    for i in range(nSol):
        beta1, beta2 = 2 * np.random.rand(), np.random.randn(dim)
        w, v = np.random.randn(dim), np.random.randn(dim)
        F1 = np.random.randn(dim) * np.exp(2 - Iter * (2 / MaxIter))
        F2 = w * (v**2) * np.cos((2 * np.random.rand()) * w)
        
        mbest = np.mean([s.X for s in Sol], axis=0)
        R_1 = 2 * np.random.rand() - 1
        S1 = (2 * np.random.rand() - 1 + np.random.randn(dim))
        S2 = (F1 * R_1 * Sol[i].X + F2 * (1 - R_1) * Best.X)
        VEC = S2 / (S1 + 1e-10) 
        
        if np.random.rand() <= 0.5:
            if np.random.rand() > Q:
                r_idx = np.random.randint(nSol)
                x_new = Best.X + beta1 * (np.exp(beta2)) * (Sol[r_idx].X - Sol[i].X)
            else:
                x_new = beta1 * VEC - Best.X
        else:
            r1 = np.random.randint(nSol)
            sign_part = (-1) ** np.random.randint(0, 2)
            x_new = (mbest * Sol[r1].X - sign_part * Sol[i].X) / (1 + (Beta * np.random.rand()))
            
        x_new = np.clip(x_new, lb, ub)
        cost, asg, met = cost_func(x_new)
        NewSol[i] = Solution(x_new, cost, asg, met)
        
        if NewSol[i].Cost < Sol[i].Cost:
            Sol[i] = NewSol[i]
    return Sol

def run_puma_optimizer(nSol, MaxIter, lb, ub, dim, cost_func):
    Sol = []
    print("Mengevaluasi populasi inisial...")
    for i in range(nSol):
        x = lb + (ub - lb) * np.random.rand(dim)
        cost, asg, met = cost_func(x)
        Sol.append(Solution(x, cost, asg, met))
        
    Best = min(Sol, key=lambda s: s.Cost)
    Initial_Best = Solution(Best.X.copy(), Best.Cost, Best.assignments, Best.metrics)
    
    # Inisialisasi hyper-parameter adaptif PO
    UnSelected = [1, 1]
    F3_Explore, F3_Exploit = 0, 0
    Seq_Time_Explore, Seq_Time_Exploit = [1, 1, 1], [1, 1, 1]
    Seq_Cost_Explore, Seq_Cost_Exploit = [1, 1, 1], [1, 1, 1]
    PF = [0.5, 0.5, 0.3]
    PF_F3 = []
    Mega_Explor, Mega_Exploit = 0.99, 0.99
    Costs_Explor, Costs_Exploit = np.zeros(3), np.zeros(3)
    Flag_Change = 1
    
    print("\n" + "="*90)
    print("MEMULAI PROSES OPTIMASI PUMA".center(90))
    print("="*90)

    # ---------------- Unexperienced Phase ----------------
    for Iter in range(1, 4):
        Sol_Explor = exploration([Solution(s.X.copy(), s.Cost, s.assignments, s.metrics) for s in Sol], lb, ub, dim, nSol, cost_func)
        Costs_Explor[Iter-1] = min([s.Cost for s in Sol_Explor])
        
        Sol_Exploit = exploitation([Solution(s.X.copy(), s.Cost, s.assignments, s.metrics) for s in Sol], lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func)
        Costs_Exploit[Iter-1] = min([s.Cost for s in Sol_Exploit])
        
        Sol = sorted(Sol + Sol_Explor + Sol_Exploit, key=lambda s: s.Cost)[:nSol]
        Best = Sol[0]
        
        print_iteration_log(Iter, Sol, Best)
        
    Seq_Cost_Explore[0] = abs(Initial_Best.Cost - Costs_Explor[0])
    Seq_Cost_Exploit[0] = abs(Initial_Best.Cost - Costs_Exploit[0])
    Seq_Cost_Explore[1] = abs(Costs_Explor[1] - Costs_Explor[0])
    Seq_Cost_Exploit[1] = abs(Costs_Exploit[1] - Costs_Exploit[0])
    Seq_Cost_Explore[2] = abs(Costs_Explor[2] - Costs_Explor[1])
    Seq_Cost_Exploit[2] = abs(Costs_Exploit[2] - Costs_Exploit[1])
    
    for i in range(3):
        if Seq_Cost_Explore[i] != 0: PF_F3.append(Seq_Cost_Explore[i])
        if Seq_Cost_Exploit[i] != 0: PF_F3.append(Seq_Cost_Exploit[i])
        
    # ---------------- Experienced Phase ----------------
    for Iter in range(4, MaxIter + 1):
        # Perhitungan dinamis mekanisme switch (Exploration vs Exploitation)
        F1_Explor = PF[0] * (Seq_Cost_Explore[0] / Seq_Time_Explore[0])
        F1_Exploit = PF[0] * (Seq_Cost_Exploit[0] / Seq_Time_Exploit[0])
        F2_Explor = PF[1] * (sum(Seq_Cost_Explore) / sum(Seq_Time_Explore))
        F2_Exploit = PF[1] * (sum(Seq_Cost_Exploit) / sum(Seq_Time_Exploit))
        Score_Explore = (Mega_Explor * F1_Explor) + (Mega_Explor * F2_Explor)
        Score_Exploit = (Mega_Exploit * F1_Exploit) + (Mega_Exploit * F2_Exploit)

        if Score_Explore > Score_Exploit:
            SelectFlag = 1
            Sol = exploration(Sol, lb, ub, dim, nSol, cost_func)
            Count_select = UnSelected.copy()
            UnSelected[1] += 1; UnSelected[0] = 1
            F3_Explore = PF[2]; F3_Exploit += PF[2]
            
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Explore[2], Seq_Cost_Explore[1] = Seq_Cost_Explore[1], Seq_Cost_Explore[0]
            Seq_Cost_Explore[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Explore[0] != 0: PF_F3.append(Seq_Cost_Explore[0])
            if TBest.Cost < Best.Cost: Best = TBest
        else:
            SelectFlag = 2
            Sol = exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func)
            Count_select = UnSelected.copy()
            UnSelected[0] += 1; UnSelected[1] = 1
            F3_Explore += PF[2]; F3_Exploit = PF[2]
            
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Exploit[2], Seq_Cost_Exploit[1] = Seq_Cost_Exploit[1], Seq_Cost_Exploit[0]
            Seq_Cost_Exploit[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Exploit[0] != 0: PF_F3.append(Seq_Cost_Exploit[0])
            if TBest.Cost < Best.Cost: Best = TBest
            
        if Flag_Change != SelectFlag:
            Flag_Change = SelectFlag
            Seq_Time_Explore[2], Seq_Time_Explore[1], Seq_Time_Explore[0] = Seq_Time_Explore[1], Seq_Time_Explore[0], Count_select[0]
            Seq_Time_Exploit[2], Seq_Time_Exploit[1], Seq_Time_Exploit[0] = Seq_Time_Exploit[1], Seq_Time_Exploit[0], Count_select[1]
            
        if Score_Explore < Score_Exploit:
            Mega_Explor = max((Mega_Explor - 0.01), 0.01); Mega_Exploit = 0.99
        elif Score_Explore > Score_Exploit:
            Mega_Explor = 0.99; Mega_Exploit = max((Mega_Exploit - 0.01), 0.01)
            
        print_iteration_log(Iter, Sol, Best)
        
    return Best

# ==========================================
# 4. Eksekusi Program dan Ekspor
# ==========================================
nSol = 30     
MaxIter = 50  
dim = len(mk_df_raw) # Ukuran ruang pencarian tetap sama
lb, ub = 0.0, 1.0

# Asumsi fungsi run_puma_optimizer telah di-define
best_solution = run_puma_optimizer(nSol, MaxIter, lb, ub, dim, fitness_function)

# Ekspor hasil solusi akhir ke format CSV
res_list = []
for a in best_solution.assignments:
    # Memanggil dataset mentah menggunakan index asli untuk menjaga integritas ID Event
    course = mk_df_raw.iloc[a['course_i']]
    room = rooms_df.iloc[a['room']]
    t1 = timeslots_df.iloc[a['start_slot']]
    t2 = timeslots_df.iloc[a['end_slot']]
    
    res_list.append({
        'event_id': course['event_id'], 'course_name': course['course_name'],
        'teacher_id': course['teacher_id'], 'room_id': room['room_id'],
        'day': t1['day'], 'time_start': t1['time_start'], 'time_end': t2['time_end']
    })

res_df = pd.DataFrame(res_list)
res_df.to_csv('scheduled_courses_heuristic.csv', index=False)
print("\n[INFO] Solusi eksperimen berbasis heuristik berhasil disimpan di 'scheduled_courses_heuristic.csv'")

### Heuristic Decoder & Random Key-Encoding

In [2]:
import numpy as np
import pandas as pd
import time
import copy

# ==========================================
# 1. Deklarasi & Persiapan Dataset
# ==========================================
print("[INFO] Memuat dan memproses dataset...")
mk_df_raw = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/mk_dsi_genap_praktikum_processed.csv')
rooms_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/rooms_processed.csv')
timeslots_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/timeslots_processed.csv')

valid_starts = []
for i in range(len(timeslots_df) - 1):
    t1 = timeslots_df.iloc[i]
    t2 = timeslots_df.iloc[i+1]
    if t1['day'] == t2['day'] and t1['time_end'] == t2['time_start']:
        valid_starts.append(i)

rooms_prac = rooms_df[rooms_df['is_practicum'] == True].index.tolist()
rooms_theo = rooms_df[rooms_df['is_theory'] == True].index.tolist()

# Kalkulasi base heuristic (beban dosen) untuk dikombinasikan dengan PO
teacher_workload = mk_df_raw['teacher_id'].value_counts().to_dict()
mk_df_raw['teacher_degree'] = mk_df_raw['teacher_id'].map(teacher_workload)

print("[INFO] Dataset berhasil dimuat dan diproses.")
print("[INFO] Fitur 'teacher_degree' berhasil ditambahkan ke DataFrame.")

[INFO] Memuat dan memproses dataset...
[INFO] Dataset berhasil dimuat dan diproses.
[INFO] Fitur 'teacher_degree' berhasil ditambahkan ke DataFrame.


In [3]:
# ==========================================
# 2. Random-Key Decoder & Objective Function
# ==========================================
def fitness_function(sol_x):
    assignments = []
    hc_count, hc_value = 0, 0
    sc_count, sc_value = 0, 0
    
    room_time_grid = {}
    teacher_time_grid = {}
    teacher_day_rooms = {}
    
    # 1. RANDOM KEY DECODING: POA menentukan prioritas penjadwalan
    queue = []
    for idx in range(len(mk_df_raw)):
        req_prac = mk_df_raw['requires_practicum_room'].iloc[idx]
        t_deg = mk_df_raw['teacher_degree'].iloc[idx]
        
        # Base skor heuristik untuk membantu PO
        base_score = (1000 if req_prac else 0) + (t_deg * 10)
        
        # Nilai X dari POA (0.0 - 1.0) menjadi faktor penentu urutan (dikali bobot 100)
        # Jika algoritma menemukan bottleneck, ia akan menaikkan sol_x[idx]
        final_priority = base_score + (sol_x[idx] * 100) 
        queue.append((final_priority, idx))
        
    # Urutkan berdasarkan prioritas tertinggi ke terendah
    queue.sort(key=lambda item: item[0], reverse=True)
    
    # 2. GREEDY PACKING SCHEDULER
    for priority, idx in queue:
        req_prac = mk_df_raw['requires_practicum_room'].iloc[idx]
        teacher = mk_df_raw['teacher_id'].iloc[idx]
        t_priority = mk_df_raw['teacher_priority'].iloc[idx]
        
        valid_rooms = rooms_prac if req_prac else rooms_theo
        feasible_choices = []
        
        # Cari semua slot yang 100% bebas dari hard constraints
        for r in valid_rooms:
            for start_idx in valid_starts:
                t1, t2 = start_idx, start_idx + 1
                if (r, t1) in room_time_grid or (r, t2) in room_time_grid: continue
                if (teacher, t1) in teacher_time_grid or (teacher, t2) in teacher_time_grid: continue
                feasible_choices.append((r, start_idx))
                
        if len(feasible_choices) > 0:
            # GREEDY SOFT-CONSTRAINT SELECTION
            # Pilih slot feasible yang menghasilkan SC paling kecil
            best_choice = None
            best_sc_penalty = float('inf')
            
            for (r, start_idx) in feasible_choices:
                t1, t2 = start_idx, start_idx + 1
                day = timeslots_df['day'].iloc[start_idx]
                
                temp_sc = 0
                sess1 = timeslots_df['session_type'].iloc[t1]
                sess2 = timeslots_df['session_type'].iloc[t2]
                
                if t_priority == 1 and (sess1 != 'Morning' or sess2 != 'Morning'): temp_sc += 10
                elif t_priority == 2 and (sess1 != 'Afternoon' or sess2 != 'Afternoon'): temp_sc += 10
                
                if teacher in teacher_day_rooms and day in teacher_day_rooms[teacher]:
                    if r not in teacher_day_rooms[teacher][day]:
                        temp_sc += 50 # Penalti untuk pindah ruang
                        
                if temp_sc < best_sc_penalty:
                    best_sc_penalty = temp_sc
                    best_choice = (r, start_idx, day, t1, t2)
                    if temp_sc == 0: break # Break awal jika slot sempurna ditemukan
            
            selected_r, selected_start, day, t1, t2 = best_choice
            
            # Kunci grid
            room_time_grid[(selected_r, t1)] = True
            room_time_grid[(selected_r, t2)] = True
            teacher_time_grid[(teacher, t1)] = True
            teacher_time_grid[(teacher, t2)] = True
            
            assignments.append({
                'course_i': idx, 'room': selected_r, 
                'start_slot': selected_start, 'end_slot': t2, 
                'teacher': teacher, 'priority': t_priority, 'day': day
            })
            
            sc_value += best_sc_penalty
            if teacher not in teacher_day_rooms: teacher_day_rooms[teacher] = {}
            if day not in teacher_day_rooms[teacher]: teacher_day_rooms[teacher][day] = []
            teacher_day_rooms[teacher][day].append(selected_r)
            
        else:
            # Jika tetap tidak ada slot meski urutan sudah dioptimasi = BENTROK FATAL
            hc_count += 1; hc_value += 100000  

    # Re-kalkulasi SC total untuk perpindahan ruang
    for teacher, days in teacher_day_rooms.items():
        for day, rooms in days.items():
            unique_rooms = len(set(rooms))
            if unique_rooms > 1: 
                violations = unique_rooms - 1
                sc_count += violations
                sc_value += 50 * violations
                
    total_penalty = hc_value + sc_value
    metrics = {'hc_count': hc_count, 'hc_value': hc_value, 'sc_count': sc_count, 'sc_value': sc_value}
    return total_penalty, assignments, metrics

print("[INFO] Fungsi fitness dengan random-key decoder berhasil didefinisikan.")

[INFO] Fungsi fitness dengan random-key decoder berhasil didefinisikan.


In [4]:
# ==========================================
# 3. Komponen Asli Puma Optimizer (Tidak Diubah)
# ==========================================
class Solution:
    def __init__(self, x, cost, assignments=None, metrics=None):
        self.X = x
        self.Cost = cost
        self.assignments = assignments
        self.metrics = metrics

def print_iteration_log(Iter, Sol, Best):
    worst_cost = max([s.Cost for s in Sol])
    metrics = Best.metrics
    print(f"Iter: {Iter:03d} | Best Fitness: {Best.Cost:8.0f} | Worst: {worst_cost:8.0f} || "
          f"HC Violations: {metrics['hc_count']:3d} (Val: {metrics['hc_value']:7.0f}) | "
          f"SC Val: {metrics['sc_value']:5.0f}")

def exploration(Sol, lb, ub, dim, nSol, cost_func):
    Sol = sorted(Sol, key=lambda s: s.Cost)
    pCR = 0.20
    PCR = 1 - pCR
    p = PCR / nSol
    NewSol = [None] * nSol
    for i in range(nSol):
        x = Sol[i].X
        A = list(range(nSol)); A.remove(i); np.random.shuffle(A)
        a, b, c, d, e, f = A[:6]
        G = 2 * np.random.rand() - 1
        if np.random.rand() < 0.5:
            y = np.random.rand(dim) * (ub - lb) + lb
        else:
            y = Sol[a].X + G * (Sol[a].X - Sol[b].X) + G * (((Sol[a].X - Sol[b].X) - (Sol[c].X - Sol[d].X)) + ((Sol[c].X - Sol[d].X) - (Sol[e].X - Sol[f].X)))
        y = np.clip(y, lb, ub)
        z = np.zeros(dim)
        j0 = np.random.randint(dim)
        for j in range(dim):
            z[j] = y[j] if (j == j0 or np.random.rand() <= pCR) else x[j]
        cost, asg, met = cost_func(z)
        NewSol[i] = Solution(z, cost, asg, met)
        if NewSol[i].Cost < Sol[i].Cost: Sol[i] = NewSol[i]
        else: pCR = pCR + p
    return Sol

def exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func):
    Q, Beta = 0.67, 2
    NewSol = [None] * nSol
    for i in range(nSol):
        beta1, beta2 = 2 * np.random.rand(), np.random.randn(dim)
        w, v = np.random.randn(dim), np.random.randn(dim)
        F1 = np.random.randn(dim) * np.exp(2 - Iter * (2 / MaxIter))
        F2 = w * (v**2) * np.cos((2 * np.random.rand()) * w)
        mbest = np.mean([s.X for s in Sol], axis=0)
        R_1 = 2 * np.random.rand() - 1
        S1 = (2 * np.random.rand() - 1 + np.random.randn(dim))
        S2 = (F1 * R_1 * Sol[i].X + F2 * (1 - R_1) * Best.X)
        VEC = S2 / (S1 + 1e-10) 
        if np.random.rand() <= 0.5:
            if np.random.rand() > Q:
                r_idx = np.random.randint(nSol)
                x_new = Best.X + beta1 * (np.exp(beta2)) * (Sol[r_idx].X - Sol[i].X)
            else: x_new = beta1 * VEC - Best.X
        else:
            r1 = np.random.randint(nSol)
            sign_part = (-1) ** np.random.randint(0, 2)
            x_new = (mbest * Sol[r1].X - sign_part * Sol[i].X) / (1 + (Beta * np.random.rand()))
        x_new = np.clip(x_new, lb, ub)
        cost, asg, met = cost_func(x_new)
        NewSol[i] = Solution(x_new, cost, asg, met)
        if NewSol[i].Cost < Sol[i].Cost: Sol[i] = NewSol[i]
    return Sol

def run_puma_optimizer(nSol, MaxIter, lb, ub, dim, cost_func):
    Sol = []
    print("Mengevaluasi populasi inisial...")
    for i in range(nSol):
        x = lb + (ub - lb) * np.random.rand(dim)
        cost, asg, met = cost_func(x)
        Sol.append(Solution(x, cost, asg, met))
        
    Best = min(Sol, key=lambda s: s.Cost)
    Initial_Best = Solution(Best.X.copy(), Best.Cost, Best.assignments, Best.metrics)
    
    UnSelected = [1, 1]
    F3_Explore, F3_Exploit = 0, 0
    Seq_Time_Explore, Seq_Time_Exploit = [1, 1, 1], [1, 1, 1]
    Seq_Cost_Explore, Seq_Cost_Exploit = [1, 1, 1], [1, 1, 1]
    PF = [0.5, 0.5, 0.3]
    PF_F3 = []
    Mega_Explor, Mega_Exploit = 0.99, 0.99
    Costs_Explor, Costs_Exploit = np.zeros(3), np.zeros(3)
    Flag_Change = 1
    
    print("\n" + "="*90)
    print("MEMULAI PROSES OPTIMASI PUMA".center(90))
    print("="*90)

    for Iter in range(1, 4):
        Sol_Explor = exploration([Solution(s.X.copy(), s.Cost, s.assignments, s.metrics) for s in Sol], lb, ub, dim, nSol, cost_func)
        Costs_Explor[Iter-1] = min([s.Cost for s in Sol_Explor])
        Sol_Exploit = exploitation([Solution(s.X.copy(), s.Cost, s.assignments, s.metrics) for s in Sol], lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func)
        Costs_Exploit[Iter-1] = min([s.Cost for s in Sol_Exploit])
        Sol = sorted(Sol + Sol_Explor + Sol_Exploit, key=lambda s: s.Cost)[:nSol]
        Best = Sol[0]
        print_iteration_log(Iter, Sol, Best)
        
    Seq_Cost_Explore[0] = abs(Initial_Best.Cost - Costs_Explor[0]); Seq_Cost_Exploit[0] = abs(Initial_Best.Cost - Costs_Exploit[0])
    Seq_Cost_Explore[1] = abs(Costs_Explor[1] - Costs_Explor[0]); Seq_Cost_Exploit[1] = abs(Costs_Exploit[1] - Costs_Exploit[0])
    Seq_Cost_Explore[2] = abs(Costs_Explor[2] - Costs_Explor[1]); Seq_Cost_Exploit[2] = abs(Costs_Exploit[2] - Costs_Exploit[1])
    for i in range(3):
        if Seq_Cost_Explore[i] != 0: PF_F3.append(Seq_Cost_Explore[i])
        if Seq_Cost_Exploit[i] != 0: PF_F3.append(Seq_Cost_Exploit[i])
        
    for Iter in range(4, MaxIter + 1):
        F1_Explor = PF[0] * (Seq_Cost_Explore[0] / Seq_Time_Explore[0]); F1_Exploit = PF[0] * (Seq_Cost_Exploit[0] / Seq_Time_Exploit[0])
        F2_Explor = PF[1] * (sum(Seq_Cost_Explore) / sum(Seq_Time_Explore)); F2_Exploit = PF[1] * (sum(Seq_Cost_Exploit) / sum(Seq_Time_Exploit))
        Score_Explore = (Mega_Explor * F1_Explor) + (Mega_Explor * F2_Explor)
        Score_Exploit = (Mega_Exploit * F1_Exploit) + (Mega_Exploit * F2_Exploit)

        if Score_Explore > Score_Exploit:
            SelectFlag = 1
            Sol = exploration(Sol, lb, ub, dim, nSol, cost_func)
            Count_select = UnSelected.copy()
            UnSelected[1] += 1; UnSelected[0] = 1
            F3_Explore = PF[2]; F3_Exploit += PF[2]
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Explore[2], Seq_Cost_Explore[1] = Seq_Cost_Explore[1], Seq_Cost_Explore[0]
            Seq_Cost_Explore[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Explore[0] != 0: PF_F3.append(Seq_Cost_Explore[0])
            if TBest.Cost < Best.Cost: Best = TBest
        else:
            SelectFlag = 2
            Sol = exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func)
            Count_select = UnSelected.copy()
            UnSelected[0] += 1; UnSelected[1] = 1
            F3_Explore += PF[2]; F3_Exploit = PF[2]
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Exploit[2], Seq_Cost_Exploit[1] = Seq_Cost_Exploit[1], Seq_Cost_Exploit[0]
            Seq_Cost_Exploit[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Exploit[0] != 0: PF_F3.append(Seq_Cost_Exploit[0])
            if TBest.Cost < Best.Cost: Best = TBest
            
        if Flag_Change != SelectFlag:
            Flag_Change = SelectFlag
            Seq_Time_Explore[2], Seq_Time_Explore[1], Seq_Time_Explore[0] = Seq_Time_Explore[1], Seq_Time_Explore[0], Count_select[0]
            Seq_Time_Exploit[2], Seq_Time_Exploit[1], Seq_Time_Exploit[0] = Seq_Time_Exploit[1], Seq_Time_Exploit[0], Count_select[1]
            
        if Score_Explore < Score_Exploit: Mega_Explor = max((Mega_Explor - 0.01), 0.01); Mega_Exploit = 0.99
        elif Score_Explore > Score_Exploit: Mega_Explor = 0.99; Mega_Exploit = max((Mega_Exploit - 0.01), 0.01)
            
        print_iteration_log(Iter, Sol, Best)
        
    return Best

print("[INFO] Komponen Puma Optimizer berhasil didefinisikan.")

[INFO] Komponen Puma Optimizer berhasil didefinisikan.


In [ ]:
# ==========================================
# 4. Eksekusi Program dan Ekspor
# ==========================================
nSol = 30     
MaxIter = 50  
dim = len(mk_df_raw) 
lb, ub = 0.0, 1.0

best_solution = run_puma_optimizer(nSol, MaxIter, lb, ub, dim, fitness_function)

res_list = []
for a in best_solution.assignments:
    course = mk_df_raw.iloc[a['course_i']]
    room = rooms_df.iloc[a['room']]
    t1 = timeslots_df.iloc[a['start_slot']]
    t2 = timeslots_df.iloc[a['end_slot']]
    
    res_list.append({
        'event_id': course['event_id'], 'course_name': course['course_name'],
        'teacher_id': course['teacher_id'], 'room_id': room['room_id'],
        'day': t1['day'], 'time_start': t1['time_start'], 'time_end': t2['time_end']
    })

res_df = pd.DataFrame(res_list)
res_df.to_csv('scheduled_courses_poa_rk.csv', index=False)
print("\n[INFO] Solusi berhasil disimpan di 'scheduled_courses_poa_rk.csv'")

### Vectorize Computational

In [6]:
import numpy as np
import pandas as pd
import time

# ==========================================
# 1. Deklarasi & Vektorisasi Preprocessing
# ==========================================
print("[INFO] Memuat dan memproses dataset dalam format Vektor C...")
mk_df_raw = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/mk_dsi_genap_praktikum_processed.csv')
rooms_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/rooms_processed.csv')
timeslots_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/timeslots_processed.csv')

# Integer Encoding untuk Dosen
teacher_list = mk_df_raw['teacher_id'].unique()
teacher_to_id = {t: i for i, t in enumerate(teacher_list)}
id_to_teacher = {i: t for i, t in enumerate(teacher_list)}
mk_df_raw['teacher_idx'] = mk_df_raw['teacher_id'].map(teacher_to_id)

# Integer Encoding untuk Hari dan Sesi
day_list = timeslots_df['day'].unique()
day_to_id = {d: i for i, d in enumerate(day_list)}
timeslots_df['day_idx'] = timeslots_df['day'].map(day_to_id)
timeslots_df['is_morning'] = timeslots_df['session_type'] == 'Morning'

# Pemetaan Array untuk Valid Starts
valid_starts_t1 = []
valid_starts_t2 = []
valid_starts_day = []

for i in range(len(timeslots_df) - 1):
    t1 = timeslots_df.iloc[i]
    t2 = timeslots_df.iloc[i+1]
    if t1['day'] == t2['day'] and t1['time_end'] == t2['time_start']:
        valid_starts_t1.append(i)
        valid_starts_t2.append(i+1)
        valid_starts_day.append(t1['day_idx'])

t1_arr = np.array(valid_starts_t1, dtype=np.int32)
t2_arr = np.array(valid_starts_t2, dtype=np.int32)
day_arr = np.array(valid_starts_day, dtype=np.int32)
num_valid_starts = len(t1_arr)

# Pemetaan Ruang
rooms_prac = np.where(rooms_df['is_practicum'] == True)[0]
rooms_theo = np.where(rooms_df['is_theory'] == True)[0]

# Pre-kalkulasi Atribut Course sebagai Array NumPy untuk akses secepat kilat (O(1))
req_prac_arr = mk_df_raw['requires_practicum_room'].values
teacher_idx_arr = mk_df_raw['teacher_idx'].values
t_priority_arr = mk_df_raw['teacher_priority'].values
t_degree_arr = mk_df_raw['teacher_id'].map(mk_df_raw['teacher_id'].value_counts()).values

num_rooms = len(rooms_df)
num_times = len(timeslots_df)
num_teachers = len(teacher_list)
num_days = len(day_list)
is_morning_arr = timeslots_df['is_morning'].values

print("[INFO] Dataset berhasil dimuat dan diproses dalam format vektor.")
print("[INFO] Semua fitur yang diperlukan telah di-vektorisasi untuk akses cepat.")

[INFO] Memuat dan memproses dataset dalam format Vektor C...
[INFO] Dataset berhasil dimuat dan diproses dalam format vektor.
[INFO] Semua fitur yang diperlukan telah di-vektorisasi untuk akses cepat.


In [7]:
# ==========================================
# 2. Vectorized Random-Key Decoder & Objective Function
# ==========================================
def fitness_function(sol_x):
    assignments = []
    hc_count, hc_value = 0, 0
    sc_count, sc_value = 0, 0
    
    # Inisialisasi State Tracker menggunakan C-Contiguous NumPy Boolean Arrays
    room_grid = np.zeros((num_rooms, num_times), dtype=bool)
    teacher_grid = np.zeros((num_teachers, num_times), dtype=bool)
    teacher_day_room = np.zeros((num_teachers, num_days, num_rooms), dtype=bool)
    
    # 1. RANDOM KEY DECODING
    # Vektorisasi operasi perhitungan prioritas (tanpa for-loop)
    base_scores = np.where(req_prac_arr, 1000, 0) + (t_degree_arr * 10)
    final_priorities = base_scores + (sol_x * 100)
    
    # np.argsort untuk pengurutan array super cepat secara descending
    queue_indices = np.argsort(-final_priorities)
    
    # 2. MATRICIAL GREEDY PACKING SCHEDULER
    for idx in queue_indices:
        req_prac = req_prac_arr[idx]
        teacher = teacher_idx_arr[idx]
        t_priority = t_priority_arr[idx]
        
        valid_r = rooms_prac if req_prac else rooms_theo
        
        # [OPERASI INTI SIMD]: Evaluasi Bentrok Ruang dan Dosen secara simultan pada seluruh matriks valid_starts
        room_conflict = room_grid[np.ix_(valid_r, t1_arr)] | room_grid[np.ix_(valid_r, t2_arr)]
        teacher_conflict = teacher_grid[teacher, t1_arr] | teacher_grid[teacher, t2_arr]
        
        # Broadcasting: Kombinasi matriks konflik ruang (R x ValidStarts) dengan konflik dosen (1 x ValidStarts)
        conflict_mask = room_conflict | teacher_conflict[np.newaxis, :]
        
        # Identifikasi indeks di mana konflik bernilai False (Ruang dan waktu kosong)
        feasible_r_idx, feasible_t_idx = np.where(~conflict_mask)
        
        if len(feasible_r_idx) > 0:
            best_sc_penalty = float('inf')
            best_choice = None
            
            # Iterasi hanya pada kandidat subset yang sudah 100% dijamin feasible
            for k in range(len(feasible_r_idx)):
                r_val = valid_r[feasible_r_idx[k]]
                t_idx_val = feasible_t_idx[k]
                
                t1, t2 = t1_arr[t_idx_val], t2_arr[t_idx_val]
                d = day_arr[t_idx_val]
                
                temp_sc = 0
                sess1_morn, sess2_morn = is_morning_arr[t1], is_morning_arr[t2]
                
                # Operasi Logika Skalar untuk Soft Constraint
                if t_priority == 1 and not (sess1_morn and sess2_morn): temp_sc += 10
                elif t_priority == 2 and (sess1_morn or sess2_morn): temp_sc += 10
                
                # Cek perpindahan ruang menggunakan agregasi boolean array (sangat cepat)
                if not teacher_day_room[teacher, d, r_val] and np.any(teacher_day_room[teacher, d, :]):
                    temp_sc += 50
                    
                if temp_sc < best_sc_penalty:
                    best_sc_penalty = temp_sc
                    best_choice = (r_val, t_idx_val, d, t1, t2)
                    if temp_sc == 0: break # Sempurna, langsung hentikan loop lokal
            
            selected_r, selected_start_idx, day, t1, t2 = best_choice
            
            # Eksekusi Pembaruan Matriks Global Tracker
            room_grid[selected_r, t1] = True
            room_grid[selected_r, t2] = True
            teacher_grid[teacher, t1] = True
            teacher_grid[teacher, t2] = True
            teacher_day_room[teacher, day, selected_r] = True
            
            assignments.append({
                'course_i': idx, 'room_idx': selected_r, 
                't1_idx': t1, 't2_idx': t2
            })
            
            sc_value += best_sc_penalty
            
        else:
            hc_count += 1; hc_value += 100000  

    # Re-kalkulasi akumulatif Soft Constraint (jumlah ruang unik per hari dikurangi 1)
    unique_rooms_per_day = np.sum(teacher_day_room, axis=2)
    room_movements = np.maximum(0, unique_rooms_per_day - 1)
    
    total_movement_violations = np.sum(room_movements)
    sc_count += total_movement_violations
    sc_value += 50 * total_movement_violations
                
    total_penalty = hc_value + sc_value
    metrics = {'hc_count': hc_count, 'hc_value': hc_value, 'sc_count': int(sc_count), 'sc_value': int(sc_value)}
    return total_penalty, assignments, metrics

In [8]:
# ==========================================
# 3. Komponen Asli Puma Optimizer (Tidak Diubah)
# ==========================================
class Solution:
    def __init__(self, x, cost, assignments=None, metrics=None):
        self.X = x
        self.Cost = cost
        self.assignments = assignments
        self.metrics = metrics

def print_iteration_log(Iter, Sol, Best):
    worst_cost = max([s.Cost for s in Sol])
    metrics = Best.metrics
    print(f"Iter: {Iter:03d} | Best Fitness: {Best.Cost:8.0f} | Worst: {worst_cost:8.0f} || "
          f"HC Violations: {metrics['hc_count']:3d} (Val: {metrics['hc_value']:7.0f}) | "
          f"SC Val: {metrics['sc_value']:5.0f}")

def exploration(Sol, lb, ub, dim, nSol, cost_func):
    Sol = sorted(Sol, key=lambda s: s.Cost)
    pCR = 0.20
    PCR = 1 - pCR
    p = PCR / nSol
    NewSol = [None] * nSol
    for i in range(nSol):
        x = Sol[i].X
        A = list(range(nSol)); A.remove(i); np.random.shuffle(A)
        a, b, c, d, e, f = A[:6]
        G = 2 * np.random.rand() - 1
        if np.random.rand() < 0.5:
            y = np.random.rand(dim) * (ub - lb) + lb
        else:
            y = Sol[a].X + G * (Sol[a].X - Sol[b].X) + G * (((Sol[a].X - Sol[b].X) - (Sol[c].X - Sol[d].X)) + ((Sol[c].X - Sol[d].X) - (Sol[e].X - Sol[f].X)))
        y = np.clip(y, lb, ub)
        z = np.zeros(dim)
        j0 = np.random.randint(dim)
        for j in range(dim):
            z[j] = y[j] if (j == j0 or np.random.rand() <= pCR) else x[j]
        cost, asg, met = cost_func(z)
        NewSol[i] = Solution(z, cost, asg, met)
        if NewSol[i].Cost < Sol[i].Cost: Sol[i] = NewSol[i]
        else: pCR = pCR + p
    return Sol

def exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func):
    Q, Beta = 0.67, 2
    NewSol = [None] * nSol
    for i in range(nSol):
        beta1, beta2 = 2 * np.random.rand(), np.random.randn(dim)
        w, v = np.random.randn(dim), np.random.randn(dim)
        F1 = np.random.randn(dim) * np.exp(2 - Iter * (2 / MaxIter))
        F2 = w * (v**2) * np.cos((2 * np.random.rand()) * w)
        mbest = np.mean([s.X for s in Sol], axis=0)
        R_1 = 2 * np.random.rand() - 1
        S1 = (2 * np.random.rand() - 1 + np.random.randn(dim))
        S2 = (F1 * R_1 * Sol[i].X + F2 * (1 - R_1) * Best.X)
        VEC = S2 / (S1 + 1e-10) 
        if np.random.rand() <= 0.5:
            if np.random.rand() > Q:
                r_idx = np.random.randint(nSol)
                x_new = Best.X + beta1 * (np.exp(beta2)) * (Sol[r_idx].X - Sol[i].X)
            else: x_new = beta1 * VEC - Best.X
        else:
            r1 = np.random.randint(nSol)
            sign_part = (-1) ** np.random.randint(0, 2)
            x_new = (mbest * Sol[r1].X - sign_part * Sol[i].X) / (1 + (Beta * np.random.rand()))
        x_new = np.clip(x_new, lb, ub)
        cost, asg, met = cost_func(x_new)
        NewSol[i] = Solution(x_new, cost, asg, met)
        if NewSol[i].Cost < Sol[i].Cost: Sol[i] = NewSol[i]
    return Sol

def run_puma_optimizer(nSol, MaxIter, lb, ub, dim, cost_func):
    Sol = []
    print("Mengevaluasi populasi inisial...")
    for i in range(nSol):
        x = lb + (ub - lb) * np.random.rand(dim)
        cost, asg, met = cost_func(x)
        Sol.append(Solution(x, cost, asg, met))
        
    Best = min(Sol, key=lambda s: s.Cost)
    Initial_Best = Solution(Best.X.copy(), Best.Cost, Best.assignments, Best.metrics)
    
    UnSelected = [1, 1]
    F3_Explore, F3_Exploit = 0, 0
    Seq_Time_Explore, Seq_Time_Exploit = [1, 1, 1], [1, 1, 1]
    Seq_Cost_Explore, Seq_Cost_Exploit = [1, 1, 1], [1, 1, 1]
    PF = [0.5, 0.5, 0.3]
    PF_F3 = []
    Mega_Explor, Mega_Exploit = 0.99, 0.99
    Costs_Explor, Costs_Exploit = np.zeros(3), np.zeros(3)
    Flag_Change = 1
    
    print("\n" + "="*90)
    print("MEMULAI PROSES OPTIMASI PUMA".center(90))
    print("="*90)

    for Iter in range(1, 4):
        Sol_Explor = exploration([Solution(s.X.copy(), s.Cost, s.assignments, s.metrics) for s in Sol], lb, ub, dim, nSol, cost_func)
        Costs_Explor[Iter-1] = min([s.Cost for s in Sol_Explor])
        Sol_Exploit = exploitation([Solution(s.X.copy(), s.Cost, s.assignments, s.metrics) for s in Sol], lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func)
        Costs_Exploit[Iter-1] = min([s.Cost for s in Sol_Exploit])
        Sol = sorted(Sol + Sol_Explor + Sol_Exploit, key=lambda s: s.Cost)[:nSol]
        Best = Sol[0]
        print_iteration_log(Iter, Sol, Best)
        
    Seq_Cost_Explore[0] = abs(Initial_Best.Cost - Costs_Explor[0]); Seq_Cost_Exploit[0] = abs(Initial_Best.Cost - Costs_Exploit[0])
    Seq_Cost_Explore[1] = abs(Costs_Explor[1] - Costs_Explor[0]); Seq_Cost_Exploit[1] = abs(Costs_Exploit[1] - Costs_Exploit[0])
    Seq_Cost_Explore[2] = abs(Costs_Explor[2] - Costs_Explor[1]); Seq_Cost_Exploit[2] = abs(Costs_Exploit[2] - Costs_Exploit[1])
    for i in range(3):
        if Seq_Cost_Explore[i] != 0: PF_F3.append(Seq_Cost_Explore[i])
        if Seq_Cost_Exploit[i] != 0: PF_F3.append(Seq_Cost_Exploit[i])
        
    for Iter in range(4, MaxIter + 1):
        F1_Explor = PF[0] * (Seq_Cost_Explore[0] / Seq_Time_Explore[0]); F1_Exploit = PF[0] * (Seq_Cost_Exploit[0] / Seq_Time_Exploit[0])
        F2_Explor = PF[1] * (sum(Seq_Cost_Explore) / sum(Seq_Time_Explore)); F2_Exploit = PF[1] * (sum(Seq_Cost_Exploit) / sum(Seq_Time_Exploit))
        Score_Explore = (Mega_Explor * F1_Explor) + (Mega_Explor * F2_Explor)
        Score_Exploit = (Mega_Exploit * F1_Exploit) + (Mega_Exploit * F2_Exploit)

        if Score_Explore > Score_Exploit:
            SelectFlag = 1
            Sol = exploration(Sol, lb, ub, dim, nSol, cost_func)
            Count_select = UnSelected.copy()
            UnSelected[1] += 1; UnSelected[0] = 1
            F3_Explore = PF[2]; F3_Exploit += PF[2]
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Explore[2], Seq_Cost_Explore[1] = Seq_Cost_Explore[1], Seq_Cost_Explore[0]
            Seq_Cost_Explore[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Explore[0] != 0: PF_F3.append(Seq_Cost_Explore[0])
            if TBest.Cost < Best.Cost: Best = TBest
        else:
            SelectFlag = 2
            Sol = exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, cost_func)
            Count_select = UnSelected.copy()
            UnSelected[0] += 1; UnSelected[1] = 1
            F3_Explore += PF[2]; F3_Exploit = PF[2]
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Exploit[2], Seq_Cost_Exploit[1] = Seq_Cost_Exploit[1], Seq_Cost_Exploit[0]
            Seq_Cost_Exploit[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Exploit[0] != 0: PF_F3.append(Seq_Cost_Exploit[0])
            if TBest.Cost < Best.Cost: Best = TBest
            
        if Flag_Change != SelectFlag:
            Flag_Change = SelectFlag
            Seq_Time_Explore[2], Seq_Time_Explore[1], Seq_Time_Explore[0] = Seq_Time_Explore[1], Seq_Time_Explore[0], Count_select[0]
            Seq_Time_Exploit[2], Seq_Time_Exploit[1], Seq_Time_Exploit[0] = Seq_Time_Exploit[1], Seq_Time_Exploit[0], Count_select[1]
            
        if Score_Explore < Score_Exploit: Mega_Explor = max((Mega_Explor - 0.01), 0.01); Mega_Exploit = 0.99
        elif Score_Explore > Score_Exploit: Mega_Explor = 0.99; Mega_Exploit = max((Mega_Exploit - 0.01), 0.01)
            
        print_iteration_log(Iter, Sol, Best)
        
    return Best

print("[INFO] Komponen Puma Optimizer berhasil didefinisikan.")

[INFO] Komponen Puma Optimizer berhasil didefinisikan.


In [ ]:
# ==========================================
# 4. Eksekusi Program dan Ekspor
# ==========================================
nSol = 30     
MaxIter = 50  
dim = len(mk_df_raw) 
lb, ub = 0.0, 1.0

best_solution = run_puma_optimizer(nSol, MaxIter, lb, ub, dim, fitness_function)

res_list = []
for a in best_solution.assignments:
    course = mk_df_raw.iloc[a['course_i']]
    room = rooms_df.iloc[a['room']]
    t1 = timeslots_df.iloc[a['start_slot']]
    t2 = timeslots_df.iloc[a['end_slot']]
    
    res_list.append({
        'event_id': course['event_id'], 'course_name': course['course_name'],
        'teacher_id': course['teacher_id'], 'room_id': room['room_id'],
        'day': t1['day'], 'time_start': t1['time_start'], 'time_end': t2['time_end']
    })

res_df = pd.DataFrame(res_list)
res_df.to_csv('scheduled_courses_poa_rk.csv', index=False)
print("\n[INFO] Solusi berhasil disimpan di 'scheduled_courses_poa_rk.csv'")

Mengevaluasi populasi inisial...

                               MEMULAI PROSES OPTIMASI PUMA                               
Iter: 001 | Best Fitness:  4004060 | Worst:  4008420 || HC Violations:  40 (Val: 4000000) | SC Val:  4060
Iter: 002 | Best Fitness:  4004060 | Worst:  4005230 || HC Violations:  40 (Val: 4000000) | SC Val:  4060
Iter: 003 | Best Fitness:  4003700 | Worst:  4004280 || HC Violations:  40 (Val: 4000000) | SC Val:  3700
Iter: 004 | Best Fitness:  4003700 | Worst:  4004280 || HC Violations:  40 (Val: 4000000) | SC Val:  3700
Iter: 005 | Best Fitness:  4003700 | Worst:  4004240 || HC Violations:  40 (Val: 4000000) | SC Val:  3700
Iter: 006 | Best Fitness:  4003700 | Worst:  4004240 || HC Violations:  40 (Val: 4000000) | SC Val:  3700
Iter: 007 | Best Fitness:  4003700 | Worst:  4004240 || HC Violations:  40 (Val: 4000000) | SC Val:  3700
Iter: 008 | Best Fitness:  4003700 | Worst:  4004240 || HC Violations:  40 (Val: 4000000) | SC Val:  3700
Iter: 009 | Best Fitness:  

KeyError: 'room'